# Title
AutoGluon Benchmark
## Purpose
Explain how the AutoGluon challenger benchmark is wired into the package helpers without forcing a training run.
## Inputs
`data/gold/listings_gold.xlsx`, the `run_autogluon_regression` helper, and the optional AutoGluon dependency.
## Outputs
An AutoGluon leaderboard, predictions, and benchmark rows under `results/benchmarks/...` when training is enabled.
## Key Takeaways
This notebook is safe to open on a minimal environment because execution is opt-in and depends on the advanced stack.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

def resolve_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / 'config.py').exists():
            return candidate
    raise FileNotFoundError('Could not find config.py in the current path ancestry')

PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import LISTINGS_GOLD, RESULTS_DIR, TARGET_VARIABLE
from elferspot_listings.modeling.challengers import run_autogluon_regression
from elferspot_listings.modeling.train import train_baseline_models

BENCHMARK_DIR = RESULTS_DIR / 'benchmarks' / '06_autogluon_benchmark'
RUN_TRAINING = False
RUN_AUTOGLUON = True
AUTOGLUON_TIME_LIMIT = 600
AUTOGLUON_PRESETS = 'best_quality'
AUTOGLUON_DYNAMIC_STACKING = None
AUTOGLUON_CLEAN_OUTPUT = False
ADVANCED_DEPENDENCIES_NOTE = 'Install the optional stack with `python -m pip install -e ".[advanced]"`.'

In [ ]:
gold_exists = LISTINGS_GOLD.exists()
metrics_path = BENCHMARK_DIR / 'metrics.json'
predictions_path = BENCHMARK_DIR / 'predictions.csv'
leaderboard_path = BENCHMARK_DIR / 'autogluon' / 'leaderboard.csv'

print(f'Gold input: {LISTINGS_GOLD}')
print(f'Gold present: {gold_exists}')
print(f'Benchmark directory: {BENCHMARK_DIR}')
print('AutoGluon helper imports are ready for opt-in training or artifact loading.')
print(ADVANCED_DEPENDENCIES_NOTE)

if gold_exists:
    gold_preview = pd.read_excel(LISTINGS_GOLD, nrows=3)
    print('\nGold preview:')
    print(gold_preview.to_string(index=False))

In [ ]:
if RUN_TRAINING and gold_exists:
    gold_df = pd.read_excel(LISTINGS_GOLD)
    if TARGET_VARIABLE not in gold_df.columns:
        raise KeyError(f'Target column {TARGET_VARIABLE!r} missing from gold dataset.')
    train_df = gold_df.sample(frac=0.8, random_state=42)
    test_df = gold_df.drop(train_df.index)
    predictor, predictions, leaderboard, metadata = run_autogluon_regression(
        train_df,
        test_df,
        target=TARGET_VARIABLE,
        output_dir=BENCHMARK_DIR,
        time_limit=AUTOGLUON_TIME_LIMIT,
        presets=AUTOGLUON_PRESETS,
        dynamic_stacking=AUTOGLUON_DYNAMIC_STACKING,
        clean_output=AUTOGLUON_CLEAN_OUTPUT,
    )
    print(f'AutoGluon fitted in {metadata["runtime_seconds"]:.1f}s with presets={metadata["presets"]}.')
    print('AutoGluon is enabled through run_autogluon_regression(...) or train_baseline_models(..., run_autogluon=True).')
elif metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
    print('Loaded existing benchmark metrics:')
    print(pd.DataFrame(metrics).T.sort_values('mae_eur').to_string())
    if predictions_path.exists():
        predictions = pd.read_csv(predictions_path)
        print('\nPrediction sample:')
        print(predictions[predictions['model_name'] == 'autogluon'].head().to_string(index=False))
    if leaderboard_path.exists():
        leaderboard = pd.read_csv(leaderboard_path)
        print('\nAutoGluon leaderboard:')
        print(leaderboard.head(10).to_string(index=False))
else:
    print('No AutoGluon outputs found yet. Keep RUN_TRAINING=False for review-only use, or enable it when advanced dependencies are installed.')